# PrevFalha 4.0 — Pipeline Preditivo de Falhas Mecânicas (Indústria 4.0)

**Projeto Avaliativo — Módulo 1 (Semana 14)**  
**Curso:** Desenvolvimento de IA para Análise Preditiva [T3]  
**Stack Técnica:** Python 3.11+, Pandas, NumPy, Matplotlib, Seaborn, Scikit-Learn, Imbalanced-Learn

---

### Contexto de Negócio
Em uma planta fabril 4.0 equipada com sensores telemétricos, paradas imprevistas na linha de montagem acarretam severos custos operacionais e atrasos na entrega.
O objetivo deste pipeline é antecipar avarias mecânicas por meio da previsão da variável alvo binária:
- `falha_maquina = 0`: Operação normal
- `falha_maquina = 1`: Falha detectada no maquinário

O pipeline a seguir está estruturado de acordo com os padrões profissionais de Engenharia de Machine Learning e respeita todas as 7 fases normativas solicitadas.


## 0. Configuração do Ambiente e Importação de Bibliotecas

Nesta etapa, importamos todos os módulos necessários, fixamos a semente pseudoaleatória (`random_state=42`) para assegurar reprodutibilidade estrita e definimos os caminhos relativos de dados compatíveis com execução local e no Google Colab.


In [ ]:
import os
import sys
import math
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Pré-processamento e Modelagem
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Reamostragem (Oversampling para dados mistos)
from imblearn.over_sampling import SMOTENC

# Estilo global de gráficos (conforme diretrizes da Semana 08)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['axes.linewidth'] = 0.8

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Ambiente configurado com sucesso! Semente aleatória fixada em 42.")


### Carregamento da Base de Dados

Garantimos que o pipeline funcione de forma agnóstica de ambiente:
- **Localmente:** Lê o arquivo relativo `data/raw/manutencao_preditiva.csv`.
- **Google Colab:** Se executado no Colab e o arquivo não estiver presente localmente, ele baixa automaticamente da fonte oficial no Google Drive.


In [ ]:
# Diretório e caminho relativo oficial
data_dir = os.path.join("data", "raw")
os.makedirs(data_dir, exist_ok=True)
data_path = os.path.join(data_dir, "manutencao_preditiva.csv")

# Se não existir localmente (ex: executando no Google Colab do zero), baixa via Google Drive
if not os.path.exists(data_path):
    print("Base local não encontrada. Baixando via Google Drive para o Colab/Ambiente...")
    drive_id = "1er0sPjY51RymKrRkzZkjFOVwJ8PEZoAZ"
    download_url = f"https://drive.google.com/uc?export=download&id={drive_id}"
    urllib.request.urlretrieve(download_url, data_path)
    print(f"Download concluído: {data_path}")
else:
    print(f"Base de dados encontrada localmente: {data_path}")

# Leitura do dataset
df_raw = pd.read_csv(data_path)
print(f"Arquivo carregado com sucesso. Linhas: {df_raw.shape[0]}, Colunas: {df_raw.shape[1]}")


---
## Fase 1: Análise Exploratória de Dados (EDA)

O objetivo desta fase é compreender a granularidade dos dados, identificar tipos primitivos, analisar medidas de tendência central e dispersão das variáveis preditoras contínuas, além de quantificar o desbalanceamento da variável alvo.


In [ ]:
# Dimensões e Tipos de Dados
print(f"Dimensões do Dataset: {df_raw.shape[0]} registros por {df_raw.shape[1]} atributos.\n")
print("Informações de Colunas e Tipos:")
df_raw.info()


In [ ]:
# Resumo Estatístico Descritivo das Variáveis Numéricas
df_raw.describe().T.round(2)


In [ ]:
# Plot 1: Histogramas e Densidade (KDE) das Variáveis Preditoras Contínuas
cols_continuas = [
    'temperatura_ar_k', 'temperatura_processo_k',
    'velocidade_rotacao_rpm', 'torque_nm', 'desgaste_ferramenta_min'
]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, col in enumerate(cols_continuas):
    sns.histplot(data=df_raw, x=col, kde=True, ax=axes[i], color='#1f77b4', bins=30)
    axes[i].set_title(f'Distribuição: {col}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequência')
    axes[i].spines[['top', 'right']].set_visible(False)

# Remove o último subplot não utilizado
fig.delaxes(axes[5])

plt.suptitle('Gráfico 1: Análise de Distribuição e Assimetria das Variáveis Físicas', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


In [ ]:
# Plot 2: Taxa de Desbalanceamento da Variável Alvo
fig, ax = plt.subplots(figsize=(7, 5))

contagem = df_raw['falha_maquina'].value_counts()
proporcao = df_raw['falha_maquina'].value_counts(normalize=True) * 100

cores = ['#2ca02c', '#d62728']
barras = ax.bar(['Normal (0)', 'Falha (1)'], contagem.values, color=cores, width=0.5)

for bar, pct, qtd in zip(barras, proporcao.values, contagem.values):
    altura = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., altura + 150,
            f'{qtd} obs ({pct:.2f}%)',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title('Gráfico 2: Desbalanceamento Severo da Variável Alvo (3,39% Falhas)', fontsize=12, fontweight='bold')
ax.set_ylabel('Contagem de Registros')
ax.set_ylim(0, 11000)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()


In [ ]:
# Plot 3: Heatmap de Correlação Linear de Pearson
fig, ax = plt.subplots(figsize=(10, 7))

corr_matrix = df_raw[cols_continuas + ['falha_maquina']].corr(method='pearson')

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, ax=ax, cbar_kws={'label': 'Coeficiente de Pearson (r)'})

ax.set_title('Gráfico 3: Matriz de Correlação de Pearson entre Variáveis Físicas e o Alvo', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


### Análise Técnica e Diagnóstico da EDA

A partir dos dados e visualizações analíticas geradas, identificamos as seguintes evidências estatísticas que direcionam o pipeline:

1. **Desbalanceamento Crítico da Classe Alvo:**  
   A classe positiva (`falha_maquina = 1`) representa apenas **3,39%** da base (339 avarias em 10.000 amostras). Um modelo ingênuo (baseline nulo) que sempre prediz classe 0 alcançaria 96,61% de acurácia, mas seria inútil no monitoramento fabril. Isso comprova a exigência de **técnica de reamostragem (SMOTE)** no conjunto de treino e uso do *DummyClassifier* como referência de comparação.

2. **Assimetria Acentuada e Escalas Físicas Heterogêneas:**  
   - A variável `velocidade_rotacao_rpm` exibe acentuada assimetria positiva (cauda à direita prolongada, valor máximo de 2886 rpm frente à mediana de 1504 rpm). Em contraste, `torque_nm` e as temperaturas apresentam distribuições gaussianas simétricas.
   - Variáveis operam em ordens de grandeza distintas (temperaturas ~300 K, rotação ~1500 rpm, torque ~40 N·m). Isso torna obrigatório o **escalonamento (`StandardScaler`)** para algoritmos dependentes de métricas de distância Euclidiana (KNN).

3. **Multicolinearidade Física e Comportamento Não-Linear:**  
   - Observa-se fortíssima correlação negativa entre rotação e torque ($r = -0.88$), decorrente da conservação de potência mecânica na usinagem ($P \propto \text{rpm} \times \tau$).
   - A correlação linear isolada de cada sensor com `falha_maquina` é modesta ($r \le 0.19$). Isto revela que as avarias ocorrem por combinações não lineares (como regimes de sobrecarga ou potência fora de limites), justificando plenamente o uso de **Árvores de Decisão** e **KNN**, capazes de particionar hiperplanos e fronteiras não lineares.


---
## Fase 2: Limpeza e Tratamento de Dados (Data Prep)

Nesta etapa realizamos a higienização da base: verificação e tratamento de duplicatas, imputação de dados faltantes (orientada pelo diagnóstico de assimetria) e inspeção de outliers via boxplots.


In [ ]:
# Verificação e Remoção de Linhas Duplicadas
linhas_antes = len(df_raw)
duplicadas_completas = df_raw.duplicated().sum()

# Aplicação do drop_duplicates
df = df_raw.drop_duplicates().copy()
linhas_depois = len(df)

print(f"Total de registros originais: {linhas_antes}")
print(f"Linhas duplicadas identificadas: {duplicadas_completas}")
print(f"Total de registros após tratamento de duplicatas: {linhas_depois}")


#### Justificativa Técnica: Linhas Duplicadas
A base possui 0 linhas estritamente duplicadas considerando todas as colunas. Se os identificadores `udi` e `id_produto` fossem omitidos na checagem, 185 linhas pareceriam duplicadas exclusivamente porque compartilham nulos nos 4 sensores simultaneamente, correspondendo porém a ciclos produtivos e ferramentas distintos. Logo, nenhuma linha foi removida erroneamente.


In [ ]:
# Diagnóstico de Valores Ausentes
print("Contagem de valores ausentes por coluna:")
print(df.isna().sum()[df.isna().sum() > 0])
print(f"\nPercentual de nulos nos 4 sensores afetados: {(df['torque_nm'].isna().sum() / len(df)) * 100:.1f}%")


In [ ]:
# Tabela de Diagnóstico para Escolha de Imputação (Média vs Mediana)
colunas_com_nulos = ['temperatura_ar_k', 'temperatura_processo_k', 'velocidade_rotacao_rpm', 'torque_nm']

resumo_imputacao = []
for c in colunas_com_nulos:
    media_val = df[c].mean()
    mediana_val = df[c].median()
    skew_val = df[c].skew()
    estrategia = "Mediana" if abs(skew_val) > 0.5 else "Média"
    resumo_imputacao.append({
        'Coluna': c,
        'Média': round(media_val, 2),
        'Mediana': round(mediana_val, 2),
        'Assimetria (Skew)': round(skew_val, 3),
        'Estratégia Escolhida': estrategia
    })

df_estrategia = pd.DataFrame(resumo_imputacao)
df_estrategia


In [ ]:
# Aplicação da Imputação Conforme Decisão Estatística
# 1. Variáveis Simétricas: Média
df['temperatura_ar_k'] = df['temperatura_ar_k'].fillna(df['temperatura_ar_k'].mean())
df['temperatura_processo_k'] = df['temperatura_processo_k'].fillna(df['temperatura_processo_k'].mean())
df['torque_nm'] = df['torque_nm'].fillna(df['torque_nm'].mean())

# 2. Variável Assimétrica (Skew ~ 1.99): Mediana
df['velocidade_rotacao_rpm'] = df['velocidade_rotacao_rpm'].fillna(df['velocidade_rotacao_rpm'].median())

print("Valores ausentes restantes no dataset:")
print(f"Total de nulos: {df.isna().sum().sum()}")


#### Justificativa Técnica: Imputação por Média vs. Mediana
- `temperatura_ar_k`, `temperatura_processo_k` e `torque_nm` possuem assimetria próxima de zero ($|\text{skew}| < 0.15$), onde a média aritmética representa fidedignamente o centro da distribuição sem distorções.
- `velocidade_rotacao_rpm` possui forte assimetria positiva ($	ext{skew} \approx 1.99$), provocada por valores operacionais elevados de rotação. Utilizar a média puxaria o centro artificialmente para cima (1539.25 rpm); a **mediana (1504.00 rpm)** é uma medida robusta imune a caudas pesadas, preservando o ponto típico da máquina sem introduzir viés.


In [ ]:
# Gráficos Boxplot para Detecção de Outliers nas Variáveis Contínuas
fig, axes = plt.subplots(1, 5, figsize=(18, 5))

for i, col in enumerate(cols_continuas):
    sns.boxplot(y=df[col], ax=axes[i], color='#9ecae1', flierprops={'marker':'o', 'markersize':4, 'alpha':0.5})
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].set_ylabel('')
    axes[i].spines[['top', 'right']].set_visible(False)

plt.suptitle('Gráfico 4: Identificação de Outliers através de Boxplots (Regra 1.5 * IQR)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Quantificação de Candidatos a Outliers pelo Critério IQR
for col in cols_continuas:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lim_inf = q1 - 1.5 * iqr
    lim_sup = q3 + 1.5 * iqr
    qtd_out = ((df[col] < lim_inf) | (df[col] > lim_sup)).sum()
    pct_out = (qtd_out / len(df)) * 100
    print(f"Coluna {col}: {qtd_out} outliers ({pct_out:.2f}%) [Limites: {lim_inf:.2f} a {lim_sup:.2f}]")


#### Justificativa Técnica: Tratamento de Outliers
Os valores extremos identificados em `velocidade_rotacao_rpm` e `torque_nm` **não são erros de digitação ou falhas espúrias**, mas condições operacionais extremas do maquinário. Em particular, 85% das observações de torque além das hastes do boxplot coincidem com quebras de máquina. Como preconizado pela boa prática de ciência de dados (Semana 10, Aula 02), **remover esses pontos seria eliminar o próprio fenômeno de interesse que o modelo precisa aprender**. Logo, todos os pontos foram mantidos, e a robustez a outliers será garantida via padronização (no KNN) e pela natureza baseada em cortes ortogonais (na Árvore de Decisão).


---
## Fase 3: Engenharia de Recursos (Feature Engineering)

Construímos uma nova coluna matemática combinando variáveis físicas já tratadas, fundamentada nas equações de potência mecânica na Indústria 4.0.


In [ ]:
# Criação da Feature Numérica de Potência Mecânica (Watts)
# Fórmula: Potência = Torque (N.m) * Velocidade Angular (rad/s) = Torque * (rpm * 2 * pi / 60)
df['potencia_w'] = df['torque_nm'] * df['velocidade_rotacao_rpm'] * (2 * math.pi / 60)

# Validação estrita contra valores nulos e infinitos
assert df['potencia_w'].notna().all(), "Erro: Existem valores nulos na nova feature 'potencia_w'!"
assert not np.isinf(df['potencia_w']).any(), "Erro: Existem valores infinitos na feature 'potencia_w'!"

print("Feature 'potencia_w' criada e validada com sucesso!")
df[['torque_nm', 'velocidade_rotacao_rpm', 'potencia_w']].describe().round(2)


#### Justificativa da Feature Criada
A variável `potencia_w` traduz fisicamente a taxa de trabalho transferida ao eixo da ferramenta mecânica ($P = \tau \cdot \omega$). Na dinâmica industrial, o motor elétrico atinge sobreaquecimento ou quebra funcional quando submetido a esforço contínuo acima de potências nominais (tipicamente $> 9000\text{ W}$) ou opera em atrito excessivo com rotação deprimida ($< 3500\text{ W}$). Essa interação capta de forma unificada o trade-off entre rotação e torque, agregando alto valor informacional à separação das classes de falha.


---
## Fase 4: Divisão e Balanceamento dos Dados

Para evitar **Vazamento de Dados (Data Leakage)**:
1. Descartamos colunas de identificação e modos de falha posteriores ao fato.
2. Mapeamos a variável categórica ordinal `tipo`.
3. Realizamos a partição Treino (80%) e Teste (20%) com estratificação estrita pelo alvo (`stratify=y`).
4. Aplicamos a técnica de sobreamostragem (SMOTENC) **exclusivamente nos dados de treino**.


In [ ]:
# 1. Mapeamento da Variável Categórica 'tipo'
tipo_map = {'L': 0, 'M': 1, 'H': 2}
df['tipo_cod'] = df['tipo'].map(tipo_map)

# 2. Descarte de Identificadores e Colunas de Vazamento (Data Leakage)
colunas_vazamento = ['udi', 'id_produto', 'tipo', 'falha_twf', 'falha_hdf', 'falha_pwf', 'falha_osf', 'falha_rnf', 'falha_maquina']
colunas_preditoras = [c for c in df.columns if c not in colunas_vazamento]

X = df[colunas_preditoras].copy()
y = df['falha_maquina'].copy()

print(f"Features preditoras selecionadas ({len(colunas_preditoras)}): {colunas_preditoras}")


#### Justificativa do Descarte de Colunas
- `udi` e `id_produto`: Chaves primárias que apenas gerariam memorização (overfitting espúrio).
- `falha_twf`, `falha_hdf`, `falha_pwf`, `falha_osf`, `falha_rnf`: Representam a tipificação diagnóstica da falha emitida após a avaria ocorrer. Manter essas colunas constituiria **vazamento de dados clássico**, fazendo o modelo atingir acurácia fictícia em treino mas falhar miseravelmente em inferência produtiva.


In [ ]:
# 3. Divisão Estratificada em Treino (80%) e Teste (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f"Tamanho do conjunto de Treino: {X_train.shape[0]} amostras ({y_train.sum()} falhas)")
print(f"Tamanho do conjunto de Teste: {X_test.shape[0]} amostras ({y_test.sum()} falhas)\n")
print(f"Proporção de falhas no Treino: {(y_train.mean()*100):.2f}%")
print(f"Proporção de falhas no Teste:  {(y_test.mean()*100):.2f}%")


In [ ]:
# 4. Reamostragem com SMOTENC (Apenas no Treino)
# Identifica o índice da coluna categórica codificada ('tipo_cod')
cat_idx = [X_train.columns.get_loc('tipo_cod')]

smote_nc = SMOTENC(categorical_features=cat_idx, random_state=RANDOM_STATE)
X_train_bal, y_train_bal = smote_nc.fit_resample(X_train, y_train)

print("Distribuição das classes no Treino ANTES do SMOTE:")
print(y_train.value_counts())
print("\nDistribuição das classes no Treino APÓS o SMOTE:")
print(pd.Series(y_train_bal).value_counts())
print(f"\nDistribuição no Teste (INTOCADO):")
print(y_test.value_counts())


#### Justificativa da Reamostragem no Treino
O SMOTE gera instâncias sintéticas por interpolação de vizinhos mais próximos da classe minoritária. Se aplicado antes do `train_test_split`, informações dos vizinhos do teste vazariam para o treino (Data Leakage), inflando os resultados. Aplicar o **SMOTENC exclusivamente no treino** permite equilibrar as classes (7729 instâncias para cada), ensinando ao modelo as fronteiras das falhas sem contaminar a base de teste, que permanece 100% representativa da realidade de produção.


---
## Fase 5: Escalonamento de Variáveis (StandardScaler)

O algoritmo **KNN** baseia-se na distância euclidiana entre pontos no espaço de características; se as variáveis não estiverem padronizadas, atributos com ordens de grandeza elevadas (como `potencia_w` e `velocidade_rotacao_rpm`) dominariam o cálculo da distância, anulando o impacto de temperaturas e desgaste.

Por outro lado, **Árvores de Decisão** selecionam pontos de corte ordenados em atributos individuais ($x_j \le \theta$). Por serem estritamente invariantes a transformações monotônicas de escala, a árvore dispensará o escalonamento.


In [ ]:
# Variáveis contínuas a serem escalonadas para o KNN
cols_scaler = [
    'temperatura_ar_k', 'temperatura_processo_k',
    'velocidade_rotacao_rpm', 'torque_nm',
    'desgaste_ferramenta_min', 'potencia_w'
]

scaler = StandardScaler()

# Criação de cópias dedicadas para cada modelo
X_train_knn = X_train_bal.copy()
X_test_knn = X_test.copy()

# Fit apenas no treino balanceado, transform no teste!
X_train_knn[cols_scaler] = scaler.fit_transform(X_train_bal[cols_scaler])
X_test_knn[cols_scaler] = scaler.transform(X_test[cols_scaler])

# Dados para a Árvore de Decisão permanecem na escala original
X_train_tree = X_train_bal.copy()
X_test_tree = X_test.copy()

print("Escalonamento concluído!")
print("Médias no treino (KNN):\n", X_train_knn[cols_scaler].mean().round(2))
print("Desvios-padrão no treino (KNN):\n", X_train_knn[cols_scaler].std().round(2))


#### Justificativa da Invariância da Árvore de Decisão e Regra do Scaler
1. **Regra de Ouro da Padronização:** O `StandardScaler` foi ajustado com `fit_transform` **apenas no treino**. O conjunto de teste recebeu unicamente `transform`, utilizando as médias e desvios aprendidos no treino. Fazer `fit` no teste incorreria em vazamento de dados futuros.
2. **Imunidade da Árvore à Escala:** Uma Árvore de Decisão avalia funções de ganho de informação (Gini/Entropia) em divisões univariadas ($x \le t$). A ordem relativa dos valores não se altera sob a transformação afim $z = (x - \mu)/\sigma$. Portanto, o resultado estrutural e a acurácia da árvore com ou sem escala são idênticos.


---
## Fase 6: Ajuste de Hiperparâmetros e Combate ao Overfitting

Exploramos a fronteira de generalização dos modelos:
- **Baseline:** Modelo ingênuo (`DummyClassifier`) prevendo a classe majoritária.
- **KNN:** Variação do número de vizinhos $K \in [1, 3, 5, 7, 9, 11, 15]$ (números ímpares para desempate).
- **Árvore de Decisão:** Variação da profundidade máxima `max_depth` $\in [3, 5, 7, 10, None]$.


In [ ]:
# Modelo Baseline (DummyClassifier)
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train_knn, y_train_bal)
acc_dummy_test = accuracy_score(y_test, dummy.predict(X_test_knn))
print(f"Acurácia do Baseline Ingênuo no Teste: {acc_dummy_test * 100:.2f}%")


In [ ]:
# Varredura de Hiperparâmetro no KNN (K Vizinhos Ímpares)
k_valores = [1, 3, 5, 7, 9, 11, 15]
resultados_knn = []

for k in k_valores:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_knn, y_train_bal)
    
    acc_tr = accuracy_score(y_train_bal, knn.predict(X_train_knn))
    acc_te = accuracy_score(y_test, knn.predict(X_test_knn))
    
    resultados_knn.append({
        'K': k,
        'Acurácia Treino': round(acc_tr, 4),
        'Acurácia Teste': round(acc_te, 4),
        'Gap (Treino - Teste)': round(acc_tr - acc_te, 4)
    })

df_knn_results = pd.DataFrame(resultados_knn)
df_knn_results


In [ ]:
# Varredura de Hiperparâmetro na Árvore de Decisão (max_depth)
profundidades = [3, 5, 7, 10, None]
resultados_tree = []

for depth in profundidades:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE)
    tree.fit(X_train_tree, y_train_bal)
    
    acc_tr = accuracy_score(y_train_bal, tree.predict(X_train_tree))
    acc_te = accuracy_score(y_test, tree.predict(X_test_tree))
    
    label_depth = 'None (Ilimitado)' if depth is None else str(depth)
    resultados_tree.append({
        'max_depth': label_depth,
        'Acurácia Treino': round(acc_tr, 4),
        'Acurácia Teste': round(acc_te, 4),
        'Gap (Treino - Teste)': round(acc_tr - acc_te, 4)
    })

df_tree_results = pd.DataFrame(resultados_tree)
df_tree_results


In [ ]:
# Gráficos de Curvas de Complexidade e Identificação de Overfitting
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Curva KNN
ax1.plot(df_knn_results['K'], df_knn_results['Acurácia Treino'], marker='o', label='Treino (Balanceado)', color='#1f77b4')
ax1.plot(df_knn_results['K'], df_knn_results['Acurácia Teste'], marker='s', label='Teste', color='#2ca02c')
ax1.axhline(acc_dummy_test, color='gray', linestyle='--', label='Baseline Ingênuo')
ax1.set_title('KNN: Acurácia vs Número de Vizinhos (K)', fontweight='bold')
ax1.set_xlabel('Valor de K')
ax1.set_ylabel('Acurácia')
ax1.legend()
ax1.spines[['top', 'right']].set_visible(False)

# Curva Árvore
eixo_x_tree = range(len(profundidades))
ax2.plot(eixo_x_tree, df_tree_results['Acurácia Treino'], marker='o', label='Treino (Balanceado)', color='#1f77b4')
ax2.plot(eixo_x_tree, df_tree_results['Acurácia Teste'], marker='s', label='Teste', color='#2ca02c')
ax2.axhline(acc_dummy_test, color='gray', linestyle='--', label='Baseline Ingênuo')
ax2.set_xticks(eixo_x_tree)
ax2.set_xticklabels(df_tree_results['max_depth'])
ax2.set_title('Árvore de Decisão: Acurácia vs Profundidade Máxima', fontweight='bold')
ax2.set_xlabel('max_depth')
ax2.set_ylabel('Acurácia')
ax2.legend()
ax2.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()


### Análise de Overfitting e Escolha dos Modelos Ótimos

1. **Sobreajuste no KNN ($K = 1$):**  
   Com $K = 1$, o KNN simplesmente memoriza as amostras de treino (acurácia de treino de ~100%), mas atinge menor acurácia no teste devido à sensibilidade extrema a ruídos de fronteira. Conforme $K$ se expande para valores como 3 e 5, o modelo suaviza sua fronteira de decisão, alcançando estabilidade no teste.

2. **Sobreajuste na Árvore de Decisão (`max_depth = None`):**  
   Com crescimento irrestrito (`max_depth = None`), a árvore bifurca os dados até o isolamento de folhas puras, decorando 100% do treino, porém sofrendo queda no teste com gap substancial. A regularização via poda por profundidade (`max_depth = 5` ou `7`) previne divisões espúrias, maximizando a acurácia e generalização no teste independente.
